# IBM Cloud - Spark + Db2 + COS Integration Test

Notebook para validar la conectividad entre:
- **PySpark** ↔ **Db2 on Cloud** (JDBC)
- **PySpark** ↔ **Cloud Object Storage** (S3A)

Servicios IBM Cloud desplegados en `us-south` / Resource Group `Default`.

## 1. Configuración de conexión

In [ ]:
# IBM Cloud Db2 connection config
DB2_CONFIG = {
    "hostname": "6667d8e9-9d4d-4ccb-ba32-21da3bb5aafc.c1ogj3sd0tgtu0lqde00.databases.appdomain.cloud",
    "port": "30376",
    "database": "bludb",
    "username": "qtn87286",
    "password": "G5VYX4VkCRqrKeNz",  # TODO: move to env var or secrets manager
}

# IBM Cloud Object Storage config
COS_CONFIG = {
    "access_key": "786065478ff34d3b84016125490a4d12",
    "secret_key": "838da9c9a9cd2521d51e856da0dd876884f8108226f5bd7c",  # TODO: move to env var
    "endpoint": "s3.us-south.cloud-object-storage.appdomain.cloud",
    "region": "us-south",
}

# Buckets (medallion architecture)
BUCKETS = {
    "raw": "datalake-raw-us-south",
    "bronze": "datalake-bronze-us-south",
    "silver": "datalake-silver-us-south",
    "gold": "datalake-gold-us-south",
}

print("Configuration loaded.")

## 2. Inicializar SparkSession con soporte S3A (COS) y JDBC (Db2)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IBM Cloud - Db2 & COS Test")
    .config("spark.hadoop.fs.s3a.access.key", COS_CONFIG["access_key"])
    .config("spark.hadoop.fs.s3a.secret.key", COS_CONFIG["secret_key"])
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{COS_CONFIG['endpoint']}")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.jars.packages", "com.ibm.db2:jcc:11.5.9.0")
    .master("local[*]")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 3. Test Db2 - Conectar via JDBC y leer tablas del sistema

In [ ]:
# Db2 JDBC URL
jdbc_url = (
    f"jdbc:db2://{DB2_CONFIG['hostname']}:{DB2_CONFIG['port']}"
    f"/{DB2_CONFIG['database']}"
    f":sslConnection=true;"
)

jdbc_properties = {
    "user": DB2_CONFIG["username"],
    "password": DB2_CONFIG["password"],
    "driver": "com.ibm.db2.jcc.DB2Driver",
}

# Read system catalog to verify connection
df_tables = (
    spark.read.jdbc(
        url=jdbc_url,
        table="(SELECT TABSCHEMA, TABNAME, TYPE FROM SYSCAT.TABLES WHERE TABSCHEMA = 'QTN87286' FETCH FIRST 20 ROWS ONLY) AS t",
        properties=jdbc_properties,
    )
)

print(f"Connected to Db2! Found {df_tables.count()} tables.")
df_tables.show(truncate=False)

## 4. Test Db2 - Crear tabla Retail de ejemplo y escribir datos

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DecimalType

# Sample retail data (matching existing schema)
schema = StructType([
    StructField("Cod_Categoria", IntegerType(), False),
    StructField("Nombre_Categoria", StringType(), False),
])

data = [
    (1, "Electrónica"), (2, "Ropa"), (3, "Hogar"),
    (4, "Alimentos"), (5, "Deportes"),
]

df_categorias = spark.createDataFrame(data, schema)
df_categorias.show()

# Write to Db2
df_categorias.write.jdbc(
    url=jdbc_url,
    table="CATEGORIAS",
    mode="overwrite",
    properties=jdbc_properties,
)
print("Table CATEGORIAS written to Db2.")

# Read back to verify
df_verify = spark.read.jdbc(url=jdbc_url, table="CATEGORIAS", properties=jdbc_properties)
print(f"Readback: {df_verify.count()} rows from Db2")
df_verify.show()

## 5. Test COS - Escribir y leer Parquet en el Data Lake

In [ ]:
# Write categorias to COS RAW bucket as CSV (raw layer)
raw_path = f"s3a://{BUCKETS['raw']}/retail/categorias/"
df_categorias.write.mode("overwrite").csv(raw_path, header=True)
print(f"Written CSV to COS: {raw_path}")

# Read back from COS
df_raw = spark.read.csv(raw_path, header=True, inferSchema=True)
print(f"Read from COS RAW: {df_raw.count()} rows")
df_raw.show()

In [ ]:
# Transform RAW → BRONZE (Parquet format with schema enforcement)
bronze_path = f"s3a://{BUCKETS['bronze']}/retail/categorias/"

df_bronze = (
    df_raw
    .withColumn("Cod_Categoria", df_raw["Cod_Categoria"].cast(IntegerType()))
    .withColumn("Nombre_Categoria", df_raw["Nombre_Categoria"].cast(StringType()))
)

df_bronze.write.mode("overwrite").parquet(bronze_path)
print(f"Written Parquet to COS BRONZE: {bronze_path}")

# Verify
df_check = spark.read.parquet(bronze_path)
print(f"Read from COS BRONZE: {df_check.count()} rows")
df_check.printSchema()
df_check.show()

## 6. Test completo: Db2 → COS pipeline (ETL end-to-end)

In [ ]:
from pyspark.sql import functions as F

# Create a larger sample: Ventas table
ventas_schema = StructType([
    StructField("Id_Venta", IntegerType(), False),
    StructField("Cod_Producto", IntegerType(), False),
    StructField("Cod_Categoria", IntegerType(), False),
    StructField("Cantidad", IntegerType(), False),
    StructField("Precio_Unitario", DecimalType(10, 2), False),
])

ventas_data = [
    (1, 101, 1, 2, 1500.00), (2, 102, 1, 1, 2300.50),
    (3, 201, 2, 3, 450.00),  (4, 202, 2, 1, 890.00),
    (5, 301, 3, 5, 120.00),  (6, 301, 3, 2, 120.00),
    (7, 401, 4, 10, 35.50),  (8, 501, 5, 1, 3200.00),
]

df_ventas = spark.createDataFrame(ventas_data, ventas_schema)

# Write to Db2
df_ventas.write.jdbc(url=jdbc_url, table="VENTAS", mode="overwrite", properties=jdbc_properties)
print("VENTAS written to Db2.")

# ETL: Read from Db2 → Transform → Write to COS Gold as Parquet
df_from_db2 = spark.read.jdbc(url=jdbc_url, table="VENTAS", properties=jdbc_properties)

# Transform: join with categories, calculate totals
df_gold = (
    df_from_db2
    .join(df_verify, on="Cod_Categoria", how="inner")
    .withColumn("Total", F.col("Cantidad") * F.col("Precio_Unitario"))
    .groupBy("Nombre_Categoria")
    .agg(
        F.sum("Total").alias("Ventas_Total"),
        F.count("Id_Venta").alias("Num_Transacciones"),
        F.avg("Total").alias("Ticket_Promedio"),
    )
    .orderBy(F.desc("Ventas_Total"))
)

# Write to Gold layer
gold_path = f"s3a://{BUCKETS['gold']}/retail/ventas_por_categoria/"
df_gold.write.mode("overwrite").parquet(gold_path)

print("ETL Pipeline: Db2 → Transform → COS Gold ✓")
df_gold.show(truncate=False)

## 7. Validación final - Resumen de conectividad

In [ ]:
import pandas as pd

# Connectivity summary
tests = [
    {"Test": "Spark Session", "Status": "✅ OK", "Detail": f"v{spark.version}"},
    {"Test": "Db2 JDBC Read", "Status": "✅ OK", "Detail": f"{DB2_CONFIG['hostname']}:{DB2_CONFIG['port']}"},
    {"Test": "Db2 Write (CATEGORIAS)", "Status": "✅ OK", "Detail": f"{df_verify.count()} rows"},
    {"Test": "Db2 Write (VENTAS)", "Status": "✅ OK", "Detail": f"{df_from_db2.count()} rows"},
    {"Test": "COS RAW (CSV)", "Status": "✅ OK", "Detail": BUCKETS["raw"]},
    {"Test": "COS BRONZE (Parquet)", "Status": "✅ OK", "Detail": BUCKETS["bronze"]},
    {"Test": "COS GOLD (Parquet)", "Status": "✅ OK", "Detail": BUCKETS["gold"]},
    {"Test": "ETL Pipeline", "Status": "✅ OK", "Detail": "Db2 → Transform → COS Gold"},
]

pd.DataFrame(tests).style.set_properties(**{"text-align": "left"})

In [ ]:
spark.stop()
print("SparkSession stopped.")